# ЛР 3.1. Реализация CNN для классификации изображений на TensorFlow/Keras. Сравнение различных API. Transfer Learning

В этом занятии мы познакомимся с фреймворком TensorFlow и его высокоуровневым API Keras. Мы реализуем современную архитектуру EfficientNet для классификации изображений из датасета FLOWERS и сравним три основных способа создания моделей: Sequential, Functional и subclassing `(tf.keras.Model)`. Также мы обсудим, зачем нужен TensorFlow/Keras, если есть PyTorch.

### Цель работы

Изучить методы переноса обучения (Transfer Learning) и дообучения (Fine-tuning) сверточных нейронных сетей для задачи классификации изображений.

В работе исследуются:

- использование предобученной модели EfficientNetB0;
- заморозка и разморозка слоев;
- Transfer Learning;
- Fine-tuning;
- сравнение способов построения моделей:
    - Sequential API;
    - Functional API;
    - Subclassing API;
- анализ сходимости;
- анализ качества классификации.


# Датасет классификации цветов
(http://www.robots.ox.ac.uk/~vgg/data/flowers/102/index.html) состоит из 102 видов цветов встречаемых в Великобритании. Для каждого класса есть от 40 до 258 примеров, чего мало для обучения с нуля, поэтому в работе будет рассмотрена технология `Transfer Learning`.

In [ ]:
# Описание лабораторной работы:
# Использование предобученной модели EfficientNet для классификации 102 видов цветов.
# Демонстрация техники Transfer Learning (переноса обучения).
# Цель: построить модель, которая по изображению цветка определяет его вид.

# Шаг 1. Установка и импорт библиотек
!pip install tensorflow
!pip install -q efficientnet  # библиотека с предобученной моделью EfficientNet

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import scipy.io
import tarfile
import csv
import sys
import os

# Основные библиотеки для работы с нейросетями
import tensorflow as tf
import tensorflow.keras as keras
import tensorflow.keras.models as M
import tensorflow.keras.layers as L
import tensorflow.keras.backend as K
import tensorflow.keras.callbacks as C
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import LearningRateScheduler, ModelCheckpoint, EarlyStopping
from tensorflow.keras.callbacks import Callback
from tensorflow.keras import optimizers
from tensorflow.keras import applications
from tensorflow.keras import layers
import efficientnet.tfkeras as efn  # EfficientNet для TensorFlow 2.1

from sklearn.model_selection import train_test_split

import PIL
from PIL import ImageOps, ImageFilter

# Увеличим дефолтный размер графиков
from pylab import rcParams
rcParams['figure.figsize'] = 10, 5
%matplotlib inline

Загрузите архив датасета `FLOWERS` в среду разработки и разархивируйте

In [ ]:
# 1. СКАЧИВАЕМ ДАТАСЕТ FLOWERS (изображения + метки)
!wget -q https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz
!wget -q https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat
!tar -xzf 102flowers.tgz


In [ ]:
# Путь к файлу с метками
IMG_PATH = '/jpg'  # Папка с данными
labels_path = ('imagelabels.mat')

# 1. Загружаем файл .mat
mat = scipy.io.loadmat(labels_path)
labels = mat['labels'][0]  # Извлекаем массив меток

# 2. Создаем список всех изображений
# Изображения в архиве именованы как image_00001.jpg ... image_08189.jpg
image_ids = [f"image_{i:05d}.jpg" for i in range(1, len(labels) + 1)]

# 3. Создаем DataFrame
df = pd.DataFrame({
    'Id': image_ids,
    'Category': labels - 1  # Вычитаем 1, чтобы классы были от 0 до 101
})

# 4. Проверяем результат
print(df.head())
print(f"\nВсего изображений: {len(df)}")
print(f"Количество классов: {df['Category'].nunique()}")
print("\nРаспределение классов:")
print(df['Category'].value_counts().sort_index().head(10))

In [ ]:
df['Category'] = df['Category'].astype(str)

In [ ]:
# Показать случайные изображения из любых классов
print('Пример картинок (random sample)')
plt.figure(figsize=(12,8))

# Берем случайные 6 изображений из всего датасета
random_image = df.sample(n=6)
random_image_paths = random_image['Id'].values
random_image_cat = random_image['Category'].values

for index, path in enumerate(random_image_paths):
    im = PIL.Image.open('/content/jpg/'+path)
    plt.subplot(3,3, index+1)
    plt.imshow(im)
    plt.title('Class: '+str(random_image_cat[index]))
    plt.axis('off')
plt.show()

In [ ]:
# Настройки, которые мы будем использовать
EPOCHS               = 10          # по заданию ЛР
BATCH_SIZE           = 32          # баланс скорости и памяти (при OOM уменьшите до 16/8)
LR                   = 1e-3        # начальная скорость обучения
VAL_SPLIT            = 0.2         # доля train для валидации

CLASS_NUM            = 102
IMG_SIZE             = 224         # стандартный вход для EfficientNetB0
IMG_CHANNELS         = 3
input_shape          = (IMG_SIZE, IMG_SIZE, IMG_CHANNELS)

DATA_PATH = '/content/jpg/'
PATH = '/content'


**Задание.** Выполните разделение данных на обучающую и тестовую выборки.

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- 1. Разделение данных (train + test) ---
train_files, test_files, train_labels, test_labels = \
    train_test_split(df['Id'], df['Category'], test_size=0.2, random_state=42, stratify=df['Category'])

train_df = pd.DataFrame({'Id': train_files, 'Category': train_labels})
test_df = pd.DataFrame({'Id': test_files, 'Category': test_labels})

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

# --- 2. Разделение train на train и validation ---
train_df, val_df = train_test_split(
    train_df, test_size=VAL_SPLIT, random_state=42, stratify=train_df['Category']
)

print(f"New Train size: {len(train_df)}, Validation size: {len(val_df)}")

# --- 3. Data Augmentation ---
# ВАЖНО: НЕ используем rescale=1./255.
# tf.keras.applications.EfficientNetB0 ожидает пиксели в диапазоне [0, 255]
# и сам применяет внутренний preprocess_input.
train_datagen = ImageDataGenerator(
    rotation_range=30,
    zoom_range=[0.8, 1.2],
    brightness_range=[0.6, 1.4],
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Для validation и test — без аугментации и без rescale
test_datagen = ImageDataGenerator()

# --- 4. Создание генераторов ---
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=DATA_PATH,
    x_col="Id",
    y_col="Category",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

validation_generator = test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=DATA_PATH,
    x_col="Id",
    y_col="Category",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=DATA_PATH,
    x_col="Id",
    y_col="Category",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Генераторы созданы.")


## Исследование аугментации

Аугментация позволяет искусственно увеличить обучающую выборку.

Используются:

- поворот изображения (`rotation_range=30`);
- изменение масштаба (`zoom_range=[0.8, 1.2]`);
- изменение яркости (`brightness_range=[0.6, 1.4]`);
- сдвиг (`width/height_shift_range=0.1`);
- отражение (`horizontal_flip=True`).

**Почему аугментация уменьшает переобучение?**  
Модель видит больше вариаций одних и тех же объектов (ракурс, масштаб, яркость) и хуже запоминает конкретные пиксели обучающих фото, лучше обобщаясь на новые изображения.


In [ ]:
x, y = next(train_generator)

plt.figure(figsize=(12, 8))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    # EfficientNet принимает [0, 255], для отображения приводим к [0, 1]
    img = np.clip(x[i] / 255.0, 0, 1)
    plt.imshow(img)
    plt.axis("off")
plt.show()


# Построение модели EfficientNet

Используется предобученная модель EfficientNetB0.

Веса сети обучены на ImageNet.

На первом этапе все сверточные слои замораживаются.
Обучается только классификатор.

## Обучение и сравнение
Обучаем три модели на одинаковых данных и гиперпараметрах (Transfer Learning, замороженный EfficientNetB0).


In [ ]:
# Общая функция для загрузки EfficientNet без верхней части (head)
def get_base_model():
    # Отдельный экземпляр backbone для каждой модели (веса не шарятся)
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )
    # Замораживаем базовую модель для Transfer Learning
    base_model.trainable = False
    return base_model

# Общая функция для создания головы классификации
def build_classification_head(input_tensor, num_classes=CLASS_NUM):
    x = tf.keras.layers.GlobalAveragePooling2D()(input_tensor)
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    output = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return output

# Общая функция компиляции и обучения
def compile_and_train(model, model_name):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    start = time.time()
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=validation_generator,
        verbose=1
    )
    train_time = time.time() - start
    print(f"--- {model_name} обучена за {train_time:.1f} c ---")
    return model, history, train_time

results = {}


### 3.1. Sequential API

In [ ]:
# 1. Создаем Sequential модель
sequential_model = tf.keras.models.Sequential([
    get_base_model(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(CLASS_NUM, activation='softmax')
])

# 2. Компилируем и обучаем
sequential_model, seq_history, seq_time = compile_and_train(sequential_model, "Sequential Model")
results['Sequential'] = {'history': seq_history, 'time': seq_time}


### 3.2. Functional API

In [ ]:
# 1. Создаем Functional модель
input_layer = tf.keras.layers.Input(shape=input_shape)
base_model_func = get_base_model()
x = base_model_func(input_layer)
output = build_classification_head(x)
functional_model = tf.keras.models.Model(inputs=input_layer, outputs=output)

# 2. Компилируем и обучаем
functional_model, func_history, func_time = compile_and_train(functional_model, "Functional Model")
results['Functional'] = {'history': func_history, 'time': func_time}


### 3.3. Subclassing API

In [ ]:
# 1. Создаем Subclassing модель
class CustomModel(tf.keras.Model):
    def __init__(self, num_classes):
        super(CustomModel, self).__init__()
        self.base_model = get_base_model()
        self.global_pool = tf.keras.layers.GlobalAveragePooling2D()
        self.dense1 = tf.keras.layers.Dense(512, activation='relu')
        self.dropout = tf.keras.layers.Dropout(0.3)
        self.dense2 = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        x = self.base_model(inputs, training=False)  # backbone заморожен
        x = self.global_pool(x)
        x = self.dense1(x)
        x = self.dropout(x, training=training)
        return self.dense2(x)

subclassing_model = CustomModel(CLASS_NUM)

# 2. Строим модель, чтобы увидеть архитектуру (важно для Subclassing)
subclassing_model.build(input_shape=(None, IMG_SIZE, IMG_SIZE, IMG_CHANNELS))
subclassing_model.summary()

# 3. Компилируем и обучаем
subclassing_model, sub_history, sub_time = compile_and_train(subclassing_model, "Subclassing Model")
results['Subclassing'] = {'history': sub_history, 'time': sub_time}


### Оценка моделей на тестовых данных

In [ ]:
print("\n--- Оценка на тестовых данных ---")

seq_loss, seq_acc = sequential_model.evaluate(test_generator, verbose=0)
print(f"Sequential Model - Test Loss: {seq_loss:.4f}, Test Accuracy: {seq_acc:.4f}")

func_loss, func_acc = functional_model.evaluate(test_generator, verbose=0)
print(f"Functional Model - Test Loss: {func_loss:.4f}, Test Accuracy: {func_acc:.4f}")

sub_loss, sub_acc = subclassing_model.evaluate(test_generator, verbose=0)
print(f"Subclassing Model - Test Loss: {sub_loss:.4f}, Test Accuracy: {sub_acc:.4f}")

results['Sequential']['accuracy'] = seq_acc
results['Functional']['accuracy'] = func_acc
results['Subclassing']['accuracy'] = sub_acc

print("\nСравнение API:")
print(f"{'API':<14}{'Accuracy':>12}{'Time, s':>12}")
for name in ['Sequential', 'Functional', 'Subclassing']:
    print(f"{name:<14}{results[name]['accuracy']:>12.4f}{results[name]['time']:>12.1f}")


### Анализ и визуализация

In [ ]:
def plot_history(history, title):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_history(seq_history, "Sequential")
plot_history(func_history, "Functional")
plot_history(sub_history, "Subclassing")


Проведите три эксперимента (после основного обучения):

1. Измените learning rate (например, `1e-4` или `5e-3`).
2. Разморозьте часть слоев EfficientNet (fine-tuning) и обучите с малым LR (`1e-5`–`1e-4`).
3. Измените dropout (например, `0.1` или `0.5`).

Пример заполнения таблицы (подставьте свои числа после прогона):

| Эксперимент | Изменение | Accuracy | Вывод |
|---|---|---|---|
| 1 | LR = 1e-4 | … | Слишком малый LR замедляет рост accuracy |
| 2 | Разморозка второй половины слоев | … | Fine-tuning обычно повышает качество |
| 3 | Dropout = 0.5 | … | Сильнее регуляризация, может чуть снизить train acc |


**Задание.** Сравнение API

Таблица заполняется автоматически в ячейке оценки (по `results`). Ожидаемо все три API дают близкую accuracy, т.к. архитектура одинаковая; различия обычно в удобстве кода и времени.

| API | Accuracy | Время обучения |
|------|-----------|----------------|
| Sequential | см. вывод evaluate | см. вывод evaluate |
| Functional | см. вывод evaluate | см. вывод evaluate |
| Subclassing | см. вывод evaluate | см. вывод evaluate |


**Вопрос 1.** Какая из реализаций показала наивысший `accuracy` к концу обучения?

**Ответ:**  
При одинаковой архитектуре (EfficientNetB0 + GAP + Dense(512) + Dropout(0.3) + Dense(102)) все три API обычно дают очень близкую тестовую accuracy. Небольшие отличия связаны со случайностью аугментации, инициализации головы и порядка батчей, а не с самим способом объявления модели. Итогового «победителя» нужно взять из вывода ячейки оценки на тестовых данных.


**Вопрос 2.** В чем ключевые различия между `Sequential`, `Functional` и `Subclassing` API в Keras? Приведите примеры, когда каждый из них предпочтительнее использовать.

**Ответ:**  
- **Sequential** — линейный стек слоев. Удобен для простых пайплайнов без ветвлений.  
- **Functional** — явное соединение тензоров `Input → слои → Model`. Нужен для multi-input/multi-output, skip-connections, параллельных веток.  
- **Subclassing (`tf.keras.Model`)** — логика forward pass пишется в `call`. Максимальная гибкость (условия, циклы, сложные кастомные блоки), но сложнее отладка и сериализация.

В этой ЛР все три подхода решают одну задачу одинаково; выбор API — про удобство и сложность архитектуры, а не про «магический» прирост качества.


**Вопрос 3.** Что такое `Transfer Learning`. Почему в лабораторной работе использовался `Transfer Learning` с `EfficientNet`? Какие преимущества это дает перед обучением модели "с нуля"?

**Ответ:**  
**Transfer Learning** — перенос знаний предобученной модели (здесь EfficientNetB0 на ImageNet) на новую задачу. Backbone замораживается, обучается новая голова классификации под 102 класса цветов.

Почему здесь это уместно: в FLOWERS мало примеров на класс (десятки–сотни), а признаков «с нуля» недостаточно. Предобученные свертки уже извлекают полезные визуальные признаки, поэтому:
- быстрее сходимость;
- выше итоговая accuracy на малом датасете;
- меньше риск сильного переобучения по сравнению с обучением всей сети randomly initialized.
